# 🔄 基于 GitHub 模型的基础代理工作流（Python）

## 📋 工作流编排教程

本笔记本介绍了 Microsoft 代理框架强大的<strong>工作流构建器</strong>功能。学习如何创建复杂的多步骤代理工作流，以处理复杂的业务流程并无缝协调多个 AI 操作。

## 🎯 学习目标

### 🏗️ <strong>工作流架构</strong>
- <strong>工作流构建器</strong>：设计和编排复杂的多步骤流程
- <strong>事件驱动执行</strong>：处理工作流事件和状态转换
- <strong>可视化工作流设计</strong>：创建并可视化工作流结构
- **GitHub 模型集成**：在工作流上下文中利用 AI 模型

### 🔄 <strong>流程编排</strong>
- <strong>顺序操作</strong>：逻辑顺序串联多个代理任务
- <strong>条件逻辑</strong>：实现决策点和分支工作流
- <strong>错误处理</strong>：强健的错误恢复和工作流弹性
- <strong>状态管理</strong>：跟踪并管理工作流执行状态

### 📊 <strong>企业工作流模式</strong>
- <strong>业务流程自动化</strong>：自动化复杂的组织工作流
- <strong>多代理协调</strong>：协调多个专业代理
- <strong>可扩展执行</strong>：设计支持企业级操作的工作流
- <strong>监控与可观测性</strong>：跟踪工作流性能和结果

## ⚙️ 先决条件与设置

### 📦 <strong>所需依赖</strong>

安装带有工作流功能的代理框架：

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub 模型配置**

**环境设置 (.env 文件)：**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 <strong>企业用例</strong>

**业务流程示例：**
- <strong>客户入职</strong>：多步骤验证和设置工作流
- <strong>内容流程</strong>：自动化内容创建、审核和发布
- <strong>数据处理</strong>：带有 AI 驱动转换的 ETL 工作流
- <strong>质量保证</strong>：自动化测试和验证流程

**工作流优势：**
- 🎯 <strong>可靠性</strong>：确定性执行及错误恢复
- 📈 <strong>可扩展性</strong>：处理高容量流程自动化
- 🔍 <strong>可观测性</strong>：完整审计跟踪和监控
- 🔧 <strong>可维护性</strong>：可视化设计和模块化组件

## 🎨 工作流设计模式

### 基础工作流结构
```mermaid
graph TD
    A[开始] --> B[代理任务 1]
    B --> C{决策点}
    C -->|成功| D[代理任务 2]
    C -->|失败| E[错误处理程序]
    D --> F[结束]
    E --> F
```

**关键组件：**
- **WorkflowBuilder**：主要编排引擎
- **WorkflowEvent**：事件处理与通信
- **WorkflowViz**：工作流可视化表示与调试

让我们一起构建你的第一个智能工作流！🚀


In [ ]:
! pip install agent-framework-core -U

In [ ]:
# 🔄 Import Workflow and Agent Framework Components
# Core components for building sophisticated agent workflows

from agent_framework.openai import OpenAIChatClient    # 🤖 GitHub Models client integration
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz  # 🏗️ Workflow orchestration tools

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# 🔗 Initialize GitHub Models Chat Client for Workflow Operations
# Create the AI client that will power agents within our workflow
chat_client = OpenAIChatClient(
    base_url=os.environ.get("GITHUB_ENDPOINT"),    # 🌐 GitHub Models API endpoint
    api_key=os.environ.get("GITHUB_TOKEN"),        # 🔑 Authentication token
    model_id=os.environ.get("GITHUB_MODEL_ID")  # 🎯 Selected AI model
)

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent   = chat_client.create_agent(
        instructions=(
           REVIEWER_INSTRUCTIONS
        ),
        name=REVIEWER_NAME,
    )

front_desk_agent = chat_client.create_agent(
        instructions=(
            FRONTDESK_INSTRUCTIONS
        ),
        name=FRONTDESK_NAME,
    )

In [ ]:
workflow = WorkflowBuilder().set_start_executor(front_desk_agent).add_edge(front_desk_agent, reviewer_agent).build()

In [ ]:

print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
svg_file = viz.export(format="svg")
print(f"SVG file saved to: {svg_file}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
result =''
async for event in workflow.run_stream('I would like to go to Paris.'):
    if isinstance(event, DatabaseEvent):
        print(f"{event}")
    if isinstance(event, WorkflowEvent):
        result += str(event.data)
        # print(f"Workflow output: {event.data}")

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
